# Notebook 04 — 풀 파이프라인 통합

## 목표
- Notebook 01~03에서 만든 모든 컴포넌트 통합
- 웹캠 실시간 처리
- 경고 레벨별 알림 시각화
- FastAPI 서빙 준비

## 스킬업 포인트
- 실시간 추론 최적화 (프레임 스킵, 비동기)
- 에이전트 파이프라인 프로파일링
- 포트폴리오 데모 시나리오 설계

In [ ]:
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

import sys, time, threading
from typing import TypedDict, Optional, List
from collections import deque

import cv2
import numpy as np
from scipy.spatial import distance as dist
import mediapipe as mp
from ultralytics import YOLO
from langgraph.graph import StateGraph, END
from langchain_ollama import ChatOllama
from langchain_core.messages import HumanMessage
import matplotlib.pyplot as plt

print('모든 패키지 로드 완료')

## 1. 이전 노트북 컴포넌트 임포트

실무에서는 `tools/` `agents/` 모듈로 분리해서 관리합니다.

In [ ]:
# ── 상수 ──────────────────────────────────────
LEFT_EYE  = [362, 385, 387, 263, 373, 380]
RIGHT_EYE = [33, 160, 158, 133, 153, 144]
MOUTH     = [61, 291, 13, 14, 17, 0, 402, 178]
EAR_THRESH = 0.25; MAR_THRESH = 0.60

# ── 계산 함수 ──────────────────────────────────
def _ear(p): A=dist.euclidean(p[1],p[5]); B=dist.euclidean(p[2],p[4]); C=dist.euclidean(p[0],p[3]); return (A+B)/(2*C)
def _mar(p): A=dist.euclidean(p[1],p[7]); B=dist.euclidean(p[2],p[6]); C=dist.euclidean(p[3],p[5]); D=dist.euclidean(p[0],p[4]); return (A+B+C)/(2*D)

def _head_pose(lm, shape):
    h,w=shape[:2]
    m=np.array([[0,0,0],[0,-330,-65],[-225,170,-135],[225,170,-135],[-150,-150,-125],[150,-150,-125]],dtype=np.float64)
    p=np.array([[lm[i].x*w,lm[i].y*h] for i in [1,152,33,263,61,291]],dtype=np.float64)
    fl=w; cam=np.array([[fl,0,w/2],[0,fl,h/2],[0,0,1]],dtype=np.float64)
    ok,rv,_=cv2.solvePnP(m,p,cam,np.zeros((4,1)))
    if not ok: return None,None,None
    rm,_=cv2.Rodrigues(rv); a,*_=cv2.RQDecomp3x3(rm)
    return a[0]*360, a[1]*360, a[2]*360

# ── State ──────────────────────────────────────
class DMSState(TypedDict):
    frame: Optional[np.ndarray]; frame_id: int
    face_detected: bool
    ear: Optional[float]; mar: Optional[float]
    pitch: Optional[float]; yaw: Optional[float]; perclos: Optional[float]
    detected_objects: List[dict]
    is_drowsy: bool; is_yawning: bool; is_distracted: bool
    has_danger_obj: bool; risk_count: int
    alert_level: int; alert_reason: str; llm_message: str
    ear_history: List[float]

def initial_state(frame=None, frame_id=0) -> DMSState:
    return DMSState(frame=frame, frame_id=frame_id,
        face_detected=False, ear=None, mar=None, pitch=None, yaw=None, perclos=None,
        detected_objects=[], is_drowsy=False, is_yawning=False,
        is_distracted=False, has_danger_obj=False, risk_count=0,
        alert_level=0, alert_reason='정상', llm_message='', ear_history=[])

print('컴포넌트 로드 완료')

## 2. 최적화된 에이전트 파이프라인

실시간 처리를 위한 최적화:
- **LLM은 레벨 1 이상일 때만** 호출 (불필요한 추론 생략)
- **YOLO는 3프레임마다** 실행 (연산량 감소)
- **얼굴 감지 실패 시** 즉시 경고

In [ ]:
import os, urllib.request
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

# face_landmarker.task 자동 다운로드
MODEL_PATH = '../models/face_landmarker.task'
MODEL_URL  = ('https://storage.googleapis.com/mediapipe-models/'
              'face_landmarker/face_landmarker/float16/1/face_landmarker.task')

os.makedirs('../models', exist_ok=True)
if not os.path.exists(MODEL_PATH):
    print('face_landmarker.task 다운로드 중...')
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print('다운로드 완료')

_base   = mp_python.BaseOptions(model_asset_path=MODEL_PATH)
_opts   = vision.FaceLandmarkerOptions(
    base_options=_base,
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1,
    min_face_detection_confidence=0.5,
    min_face_presence_confidence=0.5,
    min_tracking_confidence=0.5,
)
_mp_mesh = vision.FaceLandmarker.create_from_options(_opts)
_yolo = YOLO('yolov8n.pt')
DANGEROUS = {67: 'cell phone', 73: 'book'}

def _check_ollama():
    try:
        import urllib.request as _ur
        _ur.urlopen('http://localhost:11434', timeout=1)
        return True
    except: return False

print('모델 로드 완료 (Tasks API + YOLOv8)')


# ── 에이전트 노드 (Tasks API 버전) ──────────────
def face_analysis_agent(state):
    if state['frame'] is None: return {**state, 'face_detected': False}
    rgb      = cv2.cvtColor(state['frame'], cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    res      = _mp_mesh.detect(mp_image)
    if not res.face_landmarks:
        return {**state, 'face_detected': False, 'ear': None, 'mar': None}
    lm   = res.face_landmarks[0]
    h, w = state['frame'].shape[:2]
    le=[(lm[i].x*w,lm[i].y*h) for i in LEFT_EYE]
    re=[(lm[i].x*w,lm[i].y*h) for i in RIGHT_EYE]
    mo=[(lm[i].x*w,lm[i].y*h) for i in MOUTH]
    ear=(_ear(le)+_ear(re))/2; mar=_mar(mo)
    pitch,yaw,_=_head_pose(lm,state['frame'].shape)
    hist=list(state.get('ear_history',[]))+[ear]
    hist=hist[-900:]
    perclos=sum(1 for e in hist if e<EAR_THRESH)/len(hist)
    return {**state,'face_detected':True,'ear':round(ear,4),'mar':round(mar,4),
            'pitch':round(pitch,2) if pitch else None,'yaw':round(yaw,2) if yaw else None,
            'perclos':round(perclos,4),'ear_history':hist}


def object_detection_agent(state):
    if state['frame'] is None or state['frame_id'] % 3 != 0:
        return {**state, 'detected_objects': state.get('detected_objects',[])}
    res=_yolo(state['frame'],verbose=False)[0]
    objs=[]
    for box in res.boxes:
        cid=int(box.cls[0]); conf=float(box.conf[0])
        if cid in DANGEROUS and conf>=0.45:
            x1,y1,x2,y2=map(int,box.xyxy[0])
            objs.append({'class':DANGEROUS[cid],'confidence':round(conf,3),'bbox':[x1,y1,x2,y2]})
    return {**state,'detected_objects':objs}


def state_classifier_agent(state):
    if not state['face_detected']:
        return {**state,'is_drowsy':False,'is_yawning':False,
                'is_distracted':False,'has_danger_obj':False,'risk_count':1}
    d=(state['ear'] or 1)<EAR_THRESH or (state['perclos'] or 0)>0.15
    y=(state['mar'] or 0)>MAR_THRESH
    i=abs(state['yaw'] or 0)>30 or (state['pitch'] or 0)>20
    o=len(state['detected_objects'])>0
    return {**state,'is_drowsy':d,'is_yawning':y,'is_distracted':i,
            'has_danger_obj':o,'risk_count':sum([d,y,i,o])}


def alert_manager_agent(state):
    r=state['risk_count']; perc=state['perclos'] or 0
    obj=state['detected_objects']
    if state['has_danger_obj']:  lv=3; reason=f"위험 물체: {obj[0]['class']}"
    elif perc>0.15 or r>=3:      lv=3; reason='심각한 졸음 운전'
    elif r==2:                   lv=2; reason='복합 위험 신호'
    elif r==1:                   lv=1; reason='주의 필요'
    else:                        lv=0; reason='정상'
    return {**state,'alert_level':lv,'alert_reason':reason}


def llm_reasoning_agent(state):
    if state['alert_level']==0:
        return {**state,'llm_message':'정상 운전 중입니다.'}
    tags={0:'안전',1:'주의',2:'경고',3:'위험'}
    msgs={0:'안전하게 운전 중입니다.',
          1:'주의가 필요합니다. 집중력을 유지하세요.',
          2:'위험 신호가 감지되었습니다! 잠시 휴식을 권장합니다.',
          3:'즉시 안전한 곳에 정차하세요! 매우 위험합니다!'}
    if not _check_ollama():
        return {**state,'llm_message':f"[{tags[state['alert_level']]}] {msgs[state['alert_level']]}"}
    try:
        llm=ChatOllama(model='qwen2.5:7b',temperature=0.3)
        reasons=[]
        if state['is_drowsy']:     reasons.append('졸음')
        if state['is_yawning']:    reasons.append('하품')
        if state['is_distracted']: reasons.append('전방이탈')
        if state['has_danger_obj']:reasons.append('위험물체')
        prompt=f"운전자 모니터링 시스템입니다. 경고레벨:{state['alert_level']}/3, 감지:{','.join(reasons)}. 한 문장으로 경고 메시지 생성."
        res=llm.invoke([HumanMessage(content=prompt)])
        return {**state,'llm_message':f"[LLM] {res.content}"}
    except:
        return {**state,'llm_message':f"[{tags[state['alert_level']]}] {msgs[state['alert_level']]}"}


# ── 그래프 빌드 ────────────────────────────────
def build_dms():
    g=StateGraph(DMSState)
    g.add_node('face_analysis',    face_analysis_agent)
    g.add_node('object_detection', object_detection_agent)
    g.add_node('state_classifier', state_classifier_agent)
    g.add_node('alert_manager',    alert_manager_agent)
    g.add_node('llm_reasoning',    llm_reasoning_agent)
    g.set_entry_point('face_analysis')
    g.add_edge('face_analysis','object_detection')
    g.add_edge('object_detection','state_classifier')
    g.add_edge('state_classifier','alert_manager')
    g.add_edge('alert_manager','llm_reasoning')
    g.add_edge('llm_reasoning',END)
    return g.compile()

dms_app = build_dms()
print('DMS 파이프라인 빌드 완료 (Tasks API)')

## 3. 실시간 웹캠 DMS

`q` 키로 종료합니다.

In [ ]:
ALERT_COLORS = {0:(0,200,0), 1:(0,165,255), 2:(0,100,255), 3:(0,0,255)}
ALERT_TEXTS  = {0:'NORMAL', 1:'CAUTION', 2:'WARNING', 3:'DANGER!'}

def render_overlay(frame, state: DMSState):
    """프레임 위에 DMS 정보 오버레이"""
    h, w = frame.shape[:2]
    lv    = state['alert_level']
    color = ALERT_COLORS[lv]

    # 상단 경고 배너
    overlay = frame.copy()
    cv2.rectangle(overlay, (0,0), (w,55), color, -1)
    cv2.addWeighted(overlay, 0.6, frame, 0.4, 0, frame)
    cv2.putText(frame, ALERT_TEXTS[lv], (10,40),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,255), 3)
    cv2.putText(frame, state['alert_reason'], (200,38),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,255,255), 2)

    # 지표 패널
    panel_y = 65
    metrics = []
    if state['ear']     is not None: metrics.append((f"EAR: {state['ear']:.3f}",  state['is_drowsy']))
    if state['mar']     is not None: metrics.append((f"MAR: {state['mar']:.3f}",  state['is_yawning']))
    if state['yaw']     is not None: metrics.append((f"Yaw: {state['yaw']:.1f}°", state['is_distracted']))
    if state['perclos'] is not None: metrics.append((f"PERCLOS: {state['perclos']*100:.1f}%", (state['perclos'] or 0)>0.15))

    for i, (txt, warn) in enumerate(metrics):
        col = (0,80,255) if warn else (200,255,200)
        cv2.putText(frame, txt, (10, panel_y + i*25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, col, 2)

    # LLM 메시지
    msg = state['llm_message']
    if msg:
        cv2.putText(frame, msg[:60], (10, h-15),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (200,200,200), 1)

    # 위험 물체 감지 시 강조
    for obj in state['detected_objects']:
        x1,y1,x2,y2 = obj['bbox']
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,0,255), 3)
        cv2.putText(frame, f"{obj['class']} {obj['confidence']}",
                    (x1, y1-8), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0,0,255), 2)
    return frame


def run_dms(source=0, max_frames=None):
    """
    DMS 실시간 실행
    source: 0=웹캠, 또는 영상 파일 경로
    max_frames: None=무제한, 숫자=해당 프레임만 처리
    """
    cap = cv2.VideoCapture(source)
    if not cap.isOpened():
        print(f'소스 열기 실패: {source}')
        return

    state  = initial_state()
    frame_id = 0
    fps_buffer = deque(maxlen=30)

    print('DMS 시작 — q 키로 종료')
    print(f'Ollama: {"연결됨" if _check_ollama() else "미연결 (Rule-based 모드)"}')

    while True:
        ret, frame = cap.read()
        if not ret: break

        frame = cv2.flip(frame, 1)
        t0 = time.time()

        # 에이전트 파이프라인 실행
        state = dms_app.invoke({**state, 'frame': frame, 'frame_id': frame_id})

        # FPS 계산
        fps_buffer.append(1.0 / (time.time() - t0 + 1e-6))
        fps = np.mean(fps_buffer)

        # 오버레이
        frame = render_overlay(frame, state)
        cv2.putText(frame, f'FPS: {fps:.1f}', (frame.shape[1]-100, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200,200,200), 2)

        cv2.imshow('DMS Agent', frame)
        frame_id += 1

        if max_frames and frame_id >= max_frames: break
        if cv2.waitKey(1) & 0xFF == ord('q'): break

    cap.release()
    cv2.destroyAllWindows()
    print(f'종료 — 총 {frame_id}프레임 처리')
    return state


# 실행!
final_state = run_dms(source=0)

## 4. 성능 프로파일링

In [ ]:
import time

# 각 에이전트 노드 실행 시간 측정
dummy_frame = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
test_state  = initial_state(frame=dummy_frame)

agents = [
    ('face_analysis',    face_analysis_agent),
    ('object_detection', object_detection_agent),
    ('state_classifier', state_classifier_agent),
    ('alert_manager',    alert_manager_agent),
    ('llm_reasoning',    llm_reasoning_agent),
]

print('노드별 실행 시간 측정 (10회 평균):')
print('-'*45)

results = []
current_state = test_state.copy()
for name, agent in agents:
    times = []
    for _ in range(10):
        t = time.perf_counter()
        current_state = agent(current_state)
        times.append((time.perf_counter()-t)*1000)
    avg = np.mean(times)
    results.append((name, avg))
    print(f'  {name:<22}: {avg:6.1f} ms')

total = sum(t for _,t in results)
print(f'{"":->45}')
print(f'  {"총 파이프라인":<22}: {total:6.1f} ms  ({1000/total:.1f} FPS)')

# 시각화
fig, ax = plt.subplots(figsize=(10,4))
names = [r[0] for r in results]
times_ms = [r[1] for r in results]
bars = ax.barh(names, times_ms, color='steelblue')
ax.set_xlabel('실행 시간 (ms)')
ax.set_title('DMS 에이전트 노드별 실행 시간')
for bar, t in zip(bars, times_ms):
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
            f'{t:.1f}ms', va='center', fontsize=10)
plt.tight_layout()
plt.savefig('../output/04_profiling.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. 포트폴리오 데모 시나리오

면접/포트폴리오 발표 시 보여줄 시나리오를 미리 설계합니다.

In [ ]:
print('='*55)
print(' DMS Agent — 포트폴리오 데모 시나리오')
print('='*55)
print()
print('시나리오 1: 정상 운전')
print('  → 전방 응시, EAR 정상 → NORMAL')
print()
print('시나리오 2: 졸음 감지')
print('  → 눈 천천히 감기면 EAR↓, PERCLOS↑ → WARNING')
print('  → 15프레임 이상 지속 → DANGER + 경고음')
print()
print('시나리오 3: 전방 주시 이탈')
print('  → 고개 옆으로 돌리면 Yaw > 30° → CAUTION')
print()
print('시나리오 4: 위험 물체 감지')
print('  → 카메라 앞에 핸드폰 → DANGER (즉시)')
print()
print('기술 어필 포인트:')
print('  ✓ LangGraph 멀티 에이전트 오케스트레이션')
print('  ✓ PERCLOS (자율주행 업계 표준 알고리즘)')
print('  ✓ 완전 로컬 LLM (Ollama, API 비용 0원)')
print('  ✓ 실시간 30FPS 처리')

## 정리 — 전체 프로젝트 완성

| 노트북 | 내용 | 스킬 |
|--------|------|------|
| 01 | MediaPipe, EAR/MAR/Head Pose | CV 기초 |
| 02 | PERCLOS, YOLO, 멀티 퓨전 | 신호 통합 |
| 03 | LangGraph StateGraph, Ollama | 에이전트 프레임워크 |
| 04 | 실시간 통합, 최적화, 프로파일링 | 시스템 통합 |

**다음 단계**: FastAPI 서빙 → Docker 배포